# DDQN N-in-a-Row
### Changes from v1:
- `ReplayBuffer` replaces `deque` — O(batch) sampling via numpy fancy indexing
- CNN: 3-channel input (board mask), 5x5 first kernel, 64->64, added FC layer
- `get_target` uses `torch.where` to avoid fragile `-inf` zero-out ordering
- Guard for `prev_state[other_player] is None` on very short games
- Gradient clipping (`max_norm=1.0`)
- `tau=0.02` soft update once per epoch

In [1]:
import numpy as np
import torch
import random
from collections import deque

## ReplayBuffer

In [2]:
class ReplayBuffer:
    def __init__(self, maxlen, state_shape, batch_size):
        self.maxlen = maxlen
        self.batch_size = batch_size
        self.ptr = 0
        self.size = 0

        self.states      = np.zeros((maxlen, *state_shape), dtype=np.float32)
        self.next_states = np.zeros((maxlen, *state_shape), dtype=np.float32)
        self.actions     = np.zeros(maxlen, dtype=np.int64)
        self.rewards     = np.zeros(maxlen, dtype=np.float32)
        self.dones       = np.zeros(maxlen, dtype=bool)

    def store(self, state, action, reward, next_state, done):
        self.states[self.ptr]      = state
        self.next_states[self.ptr] = next_state
        self.actions[self.ptr]     = action
        self.rewards[self.ptr]     = reward
        self.dones[self.ptr]       = done
        self.ptr  = (self.ptr + 1) % self.maxlen
        self.size = min(self.size + 1, self.maxlen)

    def sample(self, device):
        idx = np.random.randint(0, self.size, size=self.batch_size)
        return (
            torch.FloatTensor(self.states[idx]).to(device),
            torch.LongTensor(self.actions[idx]).to(device),
            self.rewards[idx],
            torch.FloatTensor(self.next_states[idx]).to(device),
            self.dones[idx],
        )

    def __len__(self):
        return self.size

## DQNAgent

In [3]:
class DQNAgent:
    def __init__(self, board_size_2D, board_size, DDQN=True, model_type='MLP',
                 gamma=0.99, alpha=1e-4, epsilon=1.0, epsilon_decay=0.9995,
                 min_epsilon=0.05, batch_size=64):

        self.rows, self.cols = board_size_2D
        self.board_size = board_size
        self.DDQN = DDQN
        self.model_type = model_type
        self.gamma = gamma
        self.alpha = alpha
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.min_epsilon = min_epsilon
        self.batch_size = batch_size

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = self.init_model()
        self.target_network = self.init_model()
        self.target_network.load_state_dict(self.model.state_dict())
        self.target_network.eval()

        # ReplayBuffer replaces deque - O(batch) sampling
        self.buffer = ReplayBuffer(40000, (board_size,), batch_size)


    def init_model(self):
        # MLP: input (batch, 2*board_size)
        if self.model_type == "MLP":
            model = torch.nn.Sequential(
                torch.nn.Linear(2 * self.board_size, 256),
                torch.nn.ReLU(),
                torch.nn.Linear(256, 256),
                torch.nn.ReLU(),
                torch.nn.Linear(256, self.board_size),
            )

        # CNN: input (batch, 3, rows, cols)
        # ch0: player1, ch1: player2, ch2: board mask (distinguishes padding from empty)
        elif self.model_type == "CNN":
            model = torch.nn.Sequential(
                torch.nn.Conv2d(3, 64, kernel_size=5, padding=2),  # 5x5: full n=5 pattern in 1 pass
                torch.nn.ReLU(),
                torch.nn.Conv2d(64, 64, kernel_size=3, padding=1), # 3x3: receptive field -> 7x7
                torch.nn.ReLU(),
                torch.nn.Conv2d(64, 32, kernel_size=3, padding=1),
                torch.nn.ReLU(),
                torch.nn.Flatten(),
                torch.nn.Linear(32 * self.rows * self.cols, self.board_size)
            )

        return model.to(self.device)


    def get_model_input(self, state):
        # TORCH PATH - from get_value/get_target, already on device
        if isinstance(state, torch.Tensor):
            board1 = (state == 1).float()
            board2 = (state == 2).float()

            if self.model_type == "MLP":
                x = torch.cat([board1, board2], dim=1)
            elif self.model_type == "CNN":
                board1 = board1.view(-1, self.rows, self.cols)
                board2 = board2.view(-1, self.rows, self.cols)
                mask   = torch.ones_like(board1)
                x = torch.stack([board1, board2, mask], dim=1)  # (b, 3, rows, cols)
            return x

        # NUMPY PATH - from get_action
        if state.ndim == 1:
            state = state[np.newaxis, :]

        board1 = (state == 1).astype(np.float32)
        board2 = (state == 2).astype(np.float32)

        if self.model_type == "MLP":
            x = np.concatenate([board1, board2], axis=1)
        elif self.model_type == "CNN":
            board1 = board1.reshape(-1, self.rows, self.cols)
            board2 = board2.reshape(-1, self.rows, self.cols)
            mask   = np.ones_like(board1)
            x = np.stack([board1, board2, mask], axis=1)  # (b, 3, rows, cols)

        return torch.FloatTensor(x).to(self.device)


    def get_action(self, state, greedy=False):
        valid_actions = (state == 0)

        if not greedy and np.random.uniform(0, 1) < self.epsilon:
            return np.random.choice(np.flatnonzero(valid_actions))

        with torch.no_grad():
            x = self.get_model_input(state)
            q_vals = self.model(x).squeeze(0).cpu().numpy()
            q_vals[~valid_actions] = -np.inf
            return np.argmax(q_vals)


    def store(self, state, action, reward, next_state, done):
        self.buffer.store(state, action, reward, next_state, done)

    def get_sample(self):
        return self.buffer.sample(self.device)


    def get_value(self, states, actions):
        x = self.get_model_input(states)
        q_vals = self.model(x)
        return q_vals[range(len(actions)), actions]


    def get_target(self, rewards, next_states, dones):
        valid_actions = (next_states == 0)

        with torch.no_grad():
            x = self.get_model_input(next_states)

            if self.DDQN:
                q_vals       = self.model(x).masked_fill(~valid_actions, float('-inf'))
                next_actions = q_vals.argmax(dim=1)
                target_vals  = self.target_network(x).masked_fill(~valid_actions, float('-inf'))
                future       = target_vals[range(len(next_actions)), next_actions]
            else:
                target_vals = self.target_network(x).masked_fill(~valid_actions, float('-inf'))
                future      = target_vals.max(dim=1).values

        rewards = torch.FloatTensor(rewards).to(self.device)
        dones   = torch.BoolTensor(dones).to(self.device)

        # torch.where avoids fragile -inf zeroing order
        target = torch.where(dones, rewards, rewards + self.gamma * future)
        return target


    def update_target_network(self, tau=0.02):
        for main_p, target_p in zip(self.model.parameters(), self.target_network.parameters()):
            target_p.data.copy_(tau * main_p.data + (1.0 - tau) * target_p.data)

    def decay_epsilon(self):
        self.epsilon = max(self.min_epsilon, self.epsilon * self.epsilon_decay)

## Board Environment

In [4]:
class Board:
    def __init__(self, board_size=(3, 3), n=3):
        self.rows, self.cols = board_size
        self.board_size = self.rows * self.cols
        self.n = n
        self.board = np.zeros(self.board_size)
        self.win_lines = self.calc_win_lines()

    def reset(self):
        self.board = np.zeros(self.board_size)
        return self.board

    def step(self, action, player):
        self.board[action] = player
        done, winner = self.is_terminal(self.board, player)
        if done and winner == player:
            return self.board, 1, done
        return self.board, 0, done

    def is_terminal(self, board, last_player):
        player_board = (board == last_player)
        done, winner = False, 0
        for line in self.win_lines:
            if player_board[line].all():
                winner = last_player
                done = True
                break
        if not done and np.count_nonzero(board == 0) == 0:
            done = True
        return done, winner

    def calc_win_lines(self):
        win_lines = []
        # rows
        for i in range(self.rows):
            for j in range(self.n - 1, self.cols):
                idx = i * self.cols + j
                win_lines.append(np.arange(idx - self.n + 1, idx + 1).tolist())
        # cols
        for j in range(self.cols):
            for i in range(self.n - 1, self.rows):
                idx = i * self.cols + j
                win_lines.append(np.arange(idx - self.cols*(self.n-1), idx+1, self.cols).tolist())
        # main diags
        for i in range(self.rows - self.n + 1):
            for j in range(self.cols - self.n + 1):
                start = i * self.cols + j
                win_lines.append([start + k*(self.cols+1) for k in range(self.n)])
        # anti diags
        for i in range(self.rows - self.n + 1):
            for j in range(self.n - 1, self.cols):
                start = i * self.cols + j
                win_lines.append([start + k*(self.cols-1) for k in range(self.n)])
        return win_lines

## Helpers

In [5]:
def flip_board(state, player):
    if player == 1:
        return state.copy()
    flipped = state.copy()
    flipped[state == 1] = 2
    flipped[state == 2] = 1
    return flipped

## Training Loop

In [ ]:
env   = Board(board_size=(9, 9), n=5)
agent = DQNAgent(
    (env.rows, env.cols),
    env.board_size,
    DDQN=True,
    alpha=1e-4,
    epsilon_decay=0.9995,
    model_type='CNN'
)

epochs          = 6001
min_buffer      = 10000
GAMES_PER_EPOCH = 20

optimizer = torch.optim.Adam(agent.model.parameters(), lr=agent.alpha)
loss_fn   = torch.nn.HuberLoss()
agent.model.train()

print(f"Training on: {agent.device}")

for e in range(epochs):
    steps = 0

    # --- collect experience ---
    for _ in range(GAMES_PER_EPOCH):
        state  = env.reset()
        done   = False
        player = 1
        prev_state  = {1: None, 2: None}
        prev_action = {1: None, 2: None}

        while not done:
            flipped_state = flip_board(state, player)
            action        = agent.get_action(flipped_state)

            if prev_state[player] is not None:
                agent.store(prev_state[player], prev_action[player], 0, flipped_state, False)

            prev_state[player]  = flipped_state
            prev_action[player] = action

            next_state, reward, done = env.step(action, player=player)
            steps += 1

            if done:
                flipped_next_player = flip_board(next_state, player)
                agent.store(flipped_state, action, reward, flipped_next_player, True)

                other_player = 2 if player == 1 else 1
                if prev_state[other_player] is not None:  # guard: other player may not have moved
                    flipped_next_other = flip_board(next_state, other_player)
                    agent.store(prev_state[other_player], prev_action[other_player], -reward, flipped_next_other, True)
                break

            state  = next_state
            player = 2 if player == 1 else 1

    # --- gradient updates ---
    if len(agent.buffer) >= min_buffer:
        for _ in range(200):
            s, a, r, ns, d = agent.get_sample()

            optimizer.zero_grad()
            value  = agent.get_value(s, a)
            target = agent.get_target(r, ns, d)

            loss = loss_fn(value, target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(agent.model.parameters(), 1.0)
            optimizer.step()

    # soft update once per epoch
    agent.update_target_network(tau=0.02)

    if e % 500 == 0:
        print(f"Epoch: {e}   Eps: {agent.epsilon:.4f}   Avg Steps: {steps / GAMES_PER_EPOCH:.1f}")
        torch.save(agent.model.state_dict(), "DQN_ninarow2.pth")

    agent.decay_epsilon()

Training on: cuda
Epoch: 0   Eps: 1.0000   Avg Steps: 54.0
Epoch: 500   Eps: 0.7788   Avg Steps: 46.3


In [ ]:
# helper

def print_board(board, board_size):

    BOARD = [" ", "X", "O"]
    rows, cols = board_size

    #for i in range(rows):
        #print(i, end=" ")
    #print()

    for i in range(rows):
        for j in range(cols):
            print(f"|{BOARD[board[i,j]]}", end="")
        print("|")
        print("-" * (2*cols+1))
        
    print()

In [ ]:
# eval

env = Board(board_size=(9,9), n=5)
agent = DQNAgent((env.rows,env.cols), env.board_size, DDQN=True, model_type='CNN')    # DQN or DDQN

# load trained model...
agent.model.load_state_dict(torch.load("DQN_ninarow2.pth"))
agent.model.eval()

board = np.zeros(env.board.size, dtype=np.int8)
rows, cols = env.rows, env.cols

print_board(board.reshape(rows,cols), board_size=(rows,cols))

while True:
    # player1 - you
    valid = False
    
    while not valid:
        row = -1
        col = -1
        while row<0 or row>=rows:
            row = int(input("enter row: "))
        while col<0 or col>=cols:
            col = int(input("enter col: "))

        flat_idx = row * cols + col
        if board[flat_idx] == 0:
            board[flat_idx] = 1
            valid = True
        else:
            print("cant place piece here - position already occupied!")

    print_board(board.reshape(rows,cols), board_size=(rows,cols))
    
    done, winner = env.is_terminal(board, last_player=1)
    if done and winner!=0:
        print(f"player {winner} won")
        break
    elif done and winner==0:
        print("draw")
        break

    # player2
    flipped_state = flip_board(board, player=2)  # Add this line
    action = agent.get_action(flipped_state, greedy=True)  # take best action
    board[action] = 2

    print_board(board.reshape(env.rows,env.cols), board_size=(env.rows,env.cols))

    done, winner = env.is_terminal(board,last_player=2)
    if done and winner!=0:
        print(f"player {winner} won")
        break
    elif done and winner==0:
        print("draw")
        break

    